# Bonus A — RAG with Frameworks

**When to use this module:** After Module 4 (Full RAG pipeline), if the group has time and is comfortable with the hand-built pipeline.

**What you'll learn:** How LangChain and LlamaIndex implement the same RAG pipeline you already built — and why knowing the internals makes using frameworks much easier.

**Time:** ~45 minutes

---

## Why frameworks?

You've already built a working RAG system from scratch. So why would you use a framework?

- **Less boilerplate** for common patterns (loaders, splitters, chains)
- **Ecosystem** of integrations — swap ChromaDB for Pinecone, Ollama for OpenAI, with one line
- **Community** of examples, recipes, and debugging advice

The risk is that frameworks abstract away the details you now understand well. Having built the pipeline by hand, you're in the best possible position to use them: you can read through a LangChain chain or a LlamaIndex query engine and understand exactly what it's doing.

We'll build the same pipeline twice — once in LangChain, once in LlamaIndex — and compare both to what you wrote in Module 4.

In [ ]:
# Install framework dependencies (not included in base requirements.txt)
# Run this cell once, then restart the kernel

# !pip install langchain langchain-community langchain-chroma
# !pip install llama-index llama-index-llms-ollama llama-index-embeddings-huggingface

---

## Part 1 — LangChain

LangChain organises RAG around three concepts:
- **Document loaders** — read files into a standard format
- **Text splitters** — chunk documents
- **Chains** — compose retrieval + generation into a single callable

![LangChain RAG chain](images/Llamaindex-Langchain.webp)

In [ ]:
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_community.llms import Ollama
from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate

from ragsst.parameters import EMBEDDING_MODEL, MODEL, DATA_PATH

In [ ]:
# 1. Load documents
loader = DirectoryLoader(DATA_PATH, glob='**/*.txt', loader_cls=TextLoader)
documents = loader.load()
print(f'Loaded {len(documents)} documents')

In [ ]:
# 2. Split into chunks
# RecursiveCharacterTextSplitter tries to split on paragraphs, then sentences, then words
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
    length_function=len,
)
chunks = splitter.split_documents(documents)
print(f'Created {len(chunks)} chunks')
print('\nSample chunk:')
print(chunks[0].page_content[:300])

In [ ]:
# 3. Embed and store in ChromaDB
embeddings = HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL)

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name='langchain_demo',
)
retriever = vectorstore.as_retriever(search_kwargs={'k': 3})
print('Vector store ready.')

In [ ]:
# 4. Define a prompt template
# Compare this to the get_context_prompt() function you wrote in Module 4
prompt_template = PromptTemplate(
    input_variables=['context', 'question'],
    template=(
        'Use the following context to answer the question. '
        'Keep the answer concise.\n\n'
        'Context:\n{context}\n\n'
        'Question: {question}\n'
        'Answer:'
    )
)

In [ ]:
# 5. Build and run the chain
llm = Ollama(model=MODEL)

qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type='stuff',           # 'stuff' = all chunks in one prompt
    retriever=retriever,
    chain_type_kwargs={'prompt': prompt_template},
    return_source_documents=True,
)

query = 'What services does the AI service center offer?'
result = qa_chain.invoke({'query': query})

print('Answer:')
print(result['result'])
print('\nSources:')
for doc in result['source_documents']:
    print(' -', doc.metadata.get('source', 'unknown'))

**Compare:** How does this differ from the pipeline you built in Module 4? The steps are identical — load, chunk, embed, retrieve, generate. LangChain just provides standard interfaces so the pieces snap together without glue code.

**To do:** Try swapping `search_kwargs={'k': 3}` to `k=5` and see if the answer changes. Then try changing `chain_type='stuff'` to `chain_type='map_reduce'` — this splits the chunks across multiple LLM calls before combining the answers. When would that be useful?

---

## Part 2 — LlamaIndex

LlamaIndex organises things slightly differently:
- **Index** — a data structure that stores and indexes your documents
- **Query engine** — takes a question, retrieves context, and generates an answer

The main difference in philosophy: LlamaIndex is more opinionated about the index structure (it has many index types — vector, tree, keyword, graph), while LangChain is more focused on composing arbitrary chains.

In [ ]:
from llama_index.core import SimpleDirectoryReader, VectorStoreIndex, Settings
from llama_index.llms.ollama import Ollama as LlamaOllama
from llama_index.embeddings.huggingface import HuggingFaceEmbedding

In [ ]:
# Configure LlamaIndex to use our local models
Settings.llm = LlamaOllama(model=MODEL, request_timeout=120.0)
Settings.embed_model = HuggingFaceEmbedding(model_name=EMBEDDING_MODEL)
Settings.chunk_size = 500
Settings.chunk_overlap = 50

In [ ]:
# 1. Load documents
reader = SimpleDirectoryReader(DATA_PATH)
documents = reader.load_data()
print(f'Loaded {len(documents)} documents')

In [ ]:
# 2. Build the index — loading, chunking, and embedding all happen here
index = VectorStoreIndex.from_documents(documents, show_progress=True)

In [ ]:
# 3. Create a query engine and ask a question
query_engine = index.as_query_engine(similarity_top_k=3)

query = 'What services does the AI service center offer?'
response = query_engine.query(query)

print('Answer:')
print(response)
print('\nSources:')
for node in response.source_nodes:
    print(f' - {node.metadata.get("file_name", "unknown")} (score: {node.score:.3f})')

**Compare:** Notice how much less code LlamaIndex needs — `from_documents()` handles chunking and embedding in one call. The tradeoff is less visibility into what's happening. With the knowledge from Module 3, you could open the LlamaIndex source and understand every step.

**To do:** Try `index.as_chat_engine()` instead of `as_query_engine()`. This gives you a multi-turn conversation over your documents — ask a follow-up question that refers to the previous answer.

---

## Part 3 — Comparison

| | Hand-built (Module 4) | LangChain | LlamaIndex |
|---|---|---|---|
| Lines of code | ~60 | ~40 | ~20 |
| Visibility | Full | Medium | Low |
| Flexibility | Total | High | Medium |
| Integrations | Manual | Many | Many |
| Best for | Learning, custom needs | Complex chains | Quick prototypes |

None of these is better — they're tools for different situations. The hand-built version is what you'd use when you need full control (custom chunking logic, unusual retrieval strategies, production systems where you need to understand every failure mode). Frameworks are what you'd use to move fast on a standard use case or prototype something quickly.

The fact that you built it by hand first means you'll never be confused by what a framework is doing under the hood.

---

## Further reading

- [LangChain RAG tutorial](https://python.langchain.com/docs/tutorials/rag/)
- [LlamaIndex starter tutorial](https://docs.llamaindex.ai/en/stable/getting_started/starter_example/)
- [LangChain vs LlamaIndex comparison](https://www.datacamp.com/blog/langchain-vs-llamaindex)